In [1]:
import os

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "3")
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import glob
import numpy as np
import pandas as pd

SEED = 42
DEBUG = False

os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)



In [2]:
import sys
import subprocess
import importlib


def _ensure_protobuf_importable():
    """
    Bugfix: ensure google.protobuf can be imported (but do NOT uninstall anything).
    """
    try:
        import google.protobuf  # noqa: F401

        return True
    except Exception as e:
        print(
            "google.protobuf not importable, attempting to install protobuf:", repr(e)
        )
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "protobuf"]
            )
            importlib.invalidate_caches()
            import google.protobuf  # noqa: F401

            print("protobuf installed and importable.")
            return True
        except Exception as e2:
            print("Warning: protobuf install failed; continuing:", repr(e2))
            return False


def _bind_getprototype(inst):
    """
    Attach GetPrototype to an *instance* if missing, mapping to GetMessageClass when available.
    Must be robust to different protobuf implementations (python/cpp).
    """
    try:
        if inst is None or hasattr(inst, "GetPrototype"):
            return False

        # Prefer instance method if present
        if hasattr(inst, "GetMessageClass") and callable(
            getattr(inst, "GetMessageClass")
        ):

            def _getproto(self, descriptor):
                return self.GetMessageClass(descriptor)

            try:
                setattr(inst, "GetPrototype", _getproto.__get__(inst, inst.__class__))
                return True
            except Exception:
                return False

        # As a last resort, if there's a pool-backed factory without GetMessageClass,
        # we can't safely emulate; return False.
        return False
    except Exception:
        return False


def _patch_messagefactory_class(MF):
    """
    Patch MessageFactory class to include GetPrototype if missing.
    """
    try:
        if MF is None or hasattr(MF, "GetPrototype"):
            return False
        if hasattr(MF, "GetMessageClass") and callable(getattr(MF, "GetMessageClass")):

            def _class_getproto(self, descriptor):
                return self.GetMessageClass(descriptor)

            setattr(MF, "GetPrototype", _class_getproto)
            return True
        return False
    except Exception:
        return False


def _patch_protobuf_messagefactory():
    """
    Bugfix: comprehensively patch protobuf factories so TensorFlow import won't crash with:
      AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

    Key fix: patch BOTH the class and any module-level singleton instances, and also
    the symbol_database.Default()._factory object which TF frequently uses.
    """
    patched_any = False

    try:
        import google.protobuf.message_factory as mf  # type: ignore

        MF = getattr(mf, "MessageFactory", None)
        if _patch_messagefactory_class(MF):
            patched_any = True
            if DEBUG:
                print("Patched: google.protobuf.message_factory.MessageFactory (class)")

        # Patch common singleton instances in message_factory
        for name in (
            "_GENERATED_MESSAGE_FACTORY",
            "_DEFAULT_FACTORY",
            "default_factory",
        ):
            inst = getattr(mf, name, None)
            if _bind_getprototype(inst):
                patched_any = True
                if DEBUG:
                    print(f"Patched: google.protobuf.message_factory.{name} (instance)")

        # Patch symbol_database.Default()._factory (critical for TF import paths)
        try:
            import google.protobuf.symbol_database as _sym_db  # type: ignore

            db = _sym_db.Default()
            inst = getattr(db, "_factory", None)
            if _bind_getprototype(inst):
                patched_any = True
                if DEBUG:
                    print(
                        "Patched: google.protobuf.symbol_database.Default()._factory (instance)"
                    )
        except Exception as e:
            if DEBUG:
                print("symbol_database patch skipped:", repr(e))

        # Also scan attributes for already-instantiated factories of the same class
        try:
            if MF is not None:
                for _, val in list(mf.__dict__.items()):
                    try:
                        if isinstance(val, MF) and _bind_getprototype(val):
                            patched_any = True
                    except Exception:
                        continue
        except Exception:
            pass

    except Exception as e:
        print("Warning: could not patch google.protobuf.message_factory:", repr(e))

    # Patch cpp_message_factory if present
    try:
        from google.protobuf.pyext import cpp_message_factory as cmf  # type: ignore

        CMF = getattr(cmf, "MessageFactory", None)
        if _patch_messagefactory_class(CMF):
            patched_any = True
            if DEBUG:
                print(
                    "Patched: google.protobuf.pyext.cpp_message_factory.MessageFactory (class)"
                )

        for name in (
            "_GENERATED_MESSAGE_FACTORY",
            "_DEFAULT_FACTORY",
            "default_factory",
        ):
            inst = getattr(cmf, name, None)
            if _bind_getprototype(inst):
                patched_any = True
                if DEBUG:
                    print(
                        f"Patched: google.protobuf.pyext.cpp_message_factory.{name} (instance)"
                    )
    except Exception as e:
        if DEBUG:
            print("cpp_message_factory patch skipped:", repr(e))

    # Patch internal python_message_factory if present
    try:
        from google.protobuf.internal import python_message_factory as pmf  # type: ignore

        PMF = getattr(pmf, "MessageFactory", None)
        if _patch_messagefactory_class(PMF):
            patched_any = True
            if DEBUG:
                print(
                    "Patched: google.protobuf.internal.python_message_factory.MessageFactory (class)"
                )

        for name in (
            "_GENERATED_MESSAGE_FACTORY",
            "_DEFAULT_FACTORY",
            "default_factory",
        ):
            inst = getattr(pmf, name, None)
            if _bind_getprototype(inst):
                patched_any = True
                if DEBUG:
                    print(
                        f"Patched: google.protobuf.internal.python_message_factory.{name} (instance)"
                    )
    except Exception as e:
        if DEBUG:
            print("python_message_factory patch skipped:", repr(e))

    return patched_any


_ensure_protobuf_importable()
_patch_protobuf_messagefactory()




False

In [3]:
def _import_tensorflow_with_retries():
    """
    Bugfix: TF import can fail depending on which protobuf backend is picked up.
    We retry after clearing modules and re-applying patches.
    """
    last_err = None
    for attempt in range(3):
        try:
            _ensure_protobuf_importable()
            _patch_protobuf_messagefactory()

            import tensorflow as tf  # noqa: F401

            return tf
        except Exception as e:
            last_err = e
            print(f"TensorFlow import failed (attempt {attempt+1}/3):", repr(e))

            # Force pure-python protobuf and clear TF/protobuf modules, then re-patch.
            os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
            os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "3"
            os.environ["TF_USE_LEGACY_KERAS"] = "1"

            for m in list(sys.modules.keys()):
                if m.startswith(("tensorflow", "google.protobuf", "protobuf", "keras")):
                    sys.modules.pop(m, None)
            importlib.invalidate_caches()

            # Re-import protobuf and patch again before the next TF import attempt
            _ensure_protobuf_importable()
            _patch_protobuf_messagefactory()

    raise RuntimeError(f"TensorFlow import failed after retries: {repr(last_err)}")


tf = _import_tensorflow_with_retries()
tf.random.set_seed(SEED)

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TF version:", tf.__version__)




2026-05-11 20:38:22.079907: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 20:38:22.107925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778531902.142446      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778531902.150722      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-11 20:38:22.186902: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [4]:
def _maybe_pip_install(wheel_path: str):
    """
    Keep original behavior: optionally install provided wheels if present.
    """
    if os.path.exists(wheel_path):
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", wheel_path]
            )
            print(f"Installed: {wheel_path}")
        except Exception as e:
            print(f"Warning: failed to install {wheel_path}: {e}")
    else:
        print(f"Wheel not found (skipping): {wheel_path}")


_maybe_pip_install(
    "/kaggle/input/kerasapplication/Keras_Applications-1.0.8-py3-none-any.whl"
)
_maybe_pip_install("/kaggle/input/efficientnet/efficientnet-1.1.1-py3-none-any.whl")



Wheel not found (skipping): /kaggle/input/kerasapplication/Keras_Applications-1.0.8-py3-none-any.whl
Wheel not found (skipping): /kaggle/input/efficientnet/efficientnet-1.1.1-py3-none-any.whl


In [5]:
my_model = None
USING_EFFICIENTNET_TFKERAS = False  # controls preprocessing choice

try:
    from efficientnet.tfkeras import EfficientNetB4 as EffNetB4_tfk  # noqa: F401

    USING_EFFICIENTNET_TFKERAS = True
    print("Imported efficientnet.tfkeras")
except Exception as e:
    print("Warning: could not import efficientnet.tfkeras:", repr(e))

candidate_weight_paths = [
    "../input/experiment-with-models-using-keras-with-updates/effnetB4_v0.25.h5",
    "/kaggle/input/experiment-with-models-using-keras-with-updates/effnetB4_v0.25.h5",
]

weight_path = None
for p in candidate_weight_paths:
    if os.path.exists(p):
        weight_path = p
        break

if weight_path is not None:
    try:
        my_model = load_model(weight_path, compile=False)
        print("Loaded model from:", weight_path)
    except Exception as e1:
        print("Warning: first load_model failed:", repr(e1))
        try:
            custom_objects = {}
            if USING_EFFICIENTIENTNET_TFKERAS:
                from efficientnet.tfkeras import EfficientNetB4 as EffB4_custom

                custom_objects["EfficientNetB4"] = EffB4_custom
            my_model = load_model(
                weight_path, compile=False, custom_objects=custom_objects
            )
            print("Loaded model from with custom_objects:", weight_path)
        except Exception as e2:
            print(
                "Warning: failed to load .h5 model, will fall back to base model:",
                repr(e2),
            )
            my_model = None

if my_model is None:
    from tensorflow.keras import Model, layers

    if USING_EFFICIENTNET_TFKERAS:
        from efficientnet.tfkeras import EfficientNetB4

        base = EfficientNetB4(
            include_top=False, weights="imagenet", input_shape=(300, 300, 3)
        )
    else:
        from tensorflow.keras.applications import EfficientNetB4

        base = EfficientNetB4(
            include_top=False, weights="imagenet", input_shape=(300, 300, 3)
        )

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(5, activation="softmax")(x)
    my_model = Model(inputs=base.input, outputs=out)
    print(
        "Built fallback EfficientNetB4(include_top=False, ImageNet weights) + 5-class head model."
    )



2026-05-11 20:38:41.006905: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


71686520/71686520 [==============================] - 2s 0us/step
Built fallback EfficientNetB4(include_top=False, ImageNet weights) + 5-class head model.


In [6]:
test_globs = [
    "../input/cassava-leaf-disease-classification/test_images/*.jpg",
    "/kaggle/input/cassava-leaf-disease-classification/test_images/*.jpg",
    "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/test_images/*.jpg",
]
test_images = []
for g in test_globs:
    test_images = glob.glob(g)
    if len(test_images) > 0:
        print("Using test glob:", g, "count:", len(test_images))
        break

if len(test_images) == 0:
    raise FileNotFoundError(
        "Could not find any test images under expected /kaggle/input paths."
    )

df_test = pd.DataFrame(test_images, columns=["path"])


def make_test_gen(batch_size=64):
    if USING_EFFICIENTNET_TFKERAS:
        from efficientnet.tfkeras import preprocess_input
    else:
        from tensorflow.keras.applications.efficientnet import preprocess_input

    my_test_idg = ImageDataGenerator(preprocessing_function=preprocess_input)
    test_gen = my_test_idg.flow_from_dataframe(
        dataframe=df_test,
        x_col="path",
        y_col=None,
        batch_size=batch_size,
        seed=SEED,
        shuffle=False,
        class_mode=None,
        target_size=(300, 300),
    )
    return test_gen




Using test glob: ../input/cassava-leaf-disease-classification/test_images/*.jpg count: 2676


In [7]:
test_gen = make_test_gen(batch_size=128)
steps = int(np.ceil(test_gen.n / test_gen.batch_size))

pred_test = my_model.predict(test_gen, steps=steps, verbose=1)
pred_test_labels = np.argmax(pred_test, axis=-1).astype(int)

final_submission = df_test.copy()
final_submission["image_id"] = final_submission["path"].str.split("/").str[-1]
final_submission["label"] = pred_test_labels

final_csv = final_submission[["image_id", "label"]]

sample_path_candidates = [
    "../input/cassava-leaf-disease-classification/sample_submission.csv",
    "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv",
    "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/sample_submission.csv",
]
sample_path = None
for p in sample_path_candidates:
    if os.path.exists(p):
        sample_path = p
        break

if sample_path is not None:
    sample_sub = pd.read_csv(sample_path)
    final_csv = sample_sub[["image_id"]].merge(final_csv, on="image_id", how="left")
    if final_csv["label"].isna().any():
        fill_label = int(pd.Series(pred_test_labels).mode().iloc[0])
        final_csv["label"] = final_csv["label"].fillna(fill_label).astype(int)

final_csv["label"] = final_csv["label"].astype(int)
final_csv.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", final_csv.shape)
print(final_csv.head())



Found 2676 validated image filenames.
21/21 [==============================] - 249s 11s/step
Wrote submission.csv with shape: (2676, 2)
         image_id  label
0  1234294272.jpg      0
1  1234332763.jpg      4
2  1234375577.jpg      4
3  1234555380.jpg      0
4  1234571117.jpg      2


In [8]:
assert os.path.exists("submission.csv"), "submission.csv was not created."
check = pd.read_csv("submission.csv")
assert list(check.columns) == [
    "image_id",
    "label",
], f"Bad columns: {check.columns.tolist()}"
assert len(check) == len(
    df_test
), f"Row count mismatch: submission={len(check)} test={len(df_test)}"
assert check["label"].between(0, 4).all(), "Labels out of range 0-4."
check.head()

,image_id,label
0,1234294272.jpg,0
1,1234332763.jpg,4
2,1234375577.jpg,4
3,1234555380.jpg,0
4,1234571117.jpg,2
